In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown
import sklearn
from sklearn.ensemble import IsolationForest

In [2]:
# %pip install scikit-learn

# file = pd.read_csv("/Users/arsh/Downloads/codex_test_data/time_series_3.csv")

In [3]:
# file['timestamp'] = pd.to_datetime(file['timestamp'])
# file.set_index('timestamp', inplace=True)


In [4]:
import glob
import os

# Path to your folder
folder = '/Users/arsh/Desktop/codex challenge/codex_test_data'

# Find all CSVs in that folder
csv_paths = sorted(glob.glob(os.path.join(folder, '*.csv')))

files = {}
for path in csv_paths:
    name = os.path.basename(path).replace('.csv', '')  # just the filename, no path
    df = pd.read_csv(path)
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')  # bad timestamps become NaT
    df = df.dropna(subset=['timestamp'])     
    df.set_index('timestamp', inplace=True)
    files[name] = df

print(f"Loaded {len(files)} files:")
for name in files.keys():
    print(f"  {name} — {len(files[name])} rows")

Loaded 14 files:
  time_series_135 — 2015 rows
  time_series_2 — 1344 rows
  time_series_3 — 1967 rows
  time_series_5 — 1344 rows
  time_series_59 — 1872 rows
  time_series_9 — 4032 rows
  timeseries_1291193_32_3193046 — 5760 rows
  timeseries_2984852_585136_10645456 — 2880 rows
  timeseries_30904_275_36336 — 2880 rows
  timeseries_3713440_321631_15757910 — 1152 rows
  timeseries_4481425_14032_20382918 — 2880 rows
  timeseries_451856_143_501480 — 1152 rows
  timeseries_451856_201_501480 — 1152 rows
  timeseries_7490128_1446925_38076080 — 2880 rows


In [ ]:
#interactive
@interact(
    file_name=Dropdown(options=list(files.keys()), description='File:'),
    window=Dropdown(options=['24h', '48h', '72h'], value='24h', description='Window:'),
    sigma=Dropdown(options=[1, 2, 3], value=2, description='Sigma:')
)
def plot_data(file_name, window, sigma):
    f = files[file_name]
    #calculate values
    rolling_mean = f['avgrtt'].rolling(window=window).mean()
    rolling_std = f['avgrtt'].rolling(window=window).std()
    z_score = (f['avgrtt'] - rolling_mean) / rolling_std

    #anomalies
    anomaly = z_score > sigma

    #plot
    plt.figure(figsize=(14, 4))
    plt.plot(f.index, f['avgrtt'], label='Average RTT', color='blue', alpha = 0.5)

    anomalies = f[anomaly]
    plt.scatter(anomalies.index, anomalies['avgrtt'], color='red', label='Anomalies', s=20, zorder=5)
    plt.xlabel('Timestamp')
    plt.ylabel('Average RTT')
    plt.title(f'Average RTT with Anomalies (Window: {window}, Sigma:{sigma})')
    plt.legend()
    plt.show()


interactive(children=(Dropdown(description='File:', options=('time_series_135', 'time_series_2', 'time_series_…

In [6]:
# comparison with isolation forest

